# POSEIDON V6 · YOLO11n / PPE 학습 Colab

목표: **T4 GPU**에서 YOLO11n 사람 감지 모델을 ONNX로 내보내고, Kaggle 산업안전/PPE 데이터셋을 내려받아 POSEIDON 전용 PPE 모델 재학습을 준비합니다.

> Kaggle 데이터셋은 라이선스와 클래스 이름을 확인한 뒤 사용하세요. 서로 다른 데이터셋을 합칠 때는 클래스 번호를 그대로 섞으면 안 됩니다.


In [ ]:
!nvidia-smi


In [ ]:
!pip -q install -U ultralytics kaggle onnx onnxslim
from ultralytics import YOLO
import os, glob, yaml, shutil


## 1. YOLO11n 사람 감지 모델 ONNX 생성
기본 COCO pretrained YOLO11n에서 `person` 클래스를 사용합니다.


In [ ]:
person_model = YOLO("yolo11n.pt")
person_onnx = person_model.export(format="onnx", imgsz=640, simplify=True, opset=17)
print("exported:", person_onnx)


## 2. Kaggle API 연결
Kaggle 계정에서 `kaggle.json` API 토큰을 받은 뒤 아래 셀에서 업로드합니다.


In [ ]:
from google.colab import files
uploaded = files.upload()
if "kaggle.json" in uploaded:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("Kaggle API ready")


## 3. 산업안전/PPE 데이터셋 다운로드
기본 예시는 `shlokraval/ppe-dataset-yolov8` 입니다. 안전모·장갑·보안경·조끼 등 산업현장 PPE 이미지가 포함됩니다. 마스크는 별도 데이터 또는 POSEIDON 현장 데이터가 필요할 수 있습니다.


In [ ]:
KAGGLE_DATASETS = [
    "shlokraval/ppe-dataset-yolov8",
]
for slug in KAGGLE_DATASETS:
    target = "/content/kaggle_ppe/" + slug.split("/")[-1]
    os.makedirs(target, exist_ok=True)
    !kaggle datasets download -d {slug} -p {target} --unzip
    print("downloaded", slug, "->", target)


## 4. data.yaml과 클래스 확인
**여기서 클래스 번호를 반드시 확인하세요.** POSEIDON 기존 PPE 클래스 순서와 Kaggle 데이터셋 순서가 다를 수 있습니다.


In [ ]:
yaml_files = glob.glob("/content/kaggle_ppe/**/data*.yaml", recursive=True) + glob.glob("/content/kaggle_ppe/**/dataset*.yaml", recursive=True)
print("YAML candidates:")
for path in yaml_files[:30]:
    print(" -", path)
    try:
        data = yaml.safe_load(open(path, encoding="utf-8"))
        print("   names:", data.get("names"))
    except Exception as e:
        print("   read error:", e)


## 5. 재학습
`DATA_YAML`을 실제 사용할 데이터셋 yaml 경로로 지정합니다. 여러 데이터셋을 합칠 때는 먼저 라벨 클래스 ID를 POSEIDON 표준으로 변환해야 합니다.


In [ ]:
DATA_YAML = ""  # 예: /content/kaggle_ppe/.../data.yaml
if DATA_YAML:
    model = YOLO("yolo11n.pt")
    model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        patience=20,
        project="/content/poseidon_runs",
        name="ppe_yolo11n",
    )
else:
    print("DATA_YAML을 지정한 뒤 실행하세요.")


## 6. 학습 결과 ONNX로 변환


In [ ]:
BEST_PT = "/content/poseidon_runs/ppe_yolo11n/weights/best.pt"
if os.path.exists(BEST_PT):
    best = YOLO(BEST_PT)
    exported = best.export(format="onnx", imgsz=640, simplify=True, opset=17)
    print("POSEIDON PPE ONNX:", exported)
else:
    print("아직 best.pt가 없습니다.")


## 다음 단계
- `yolo11n.onnx`: POSEIDON 사람 감지 모델
- `best.onnx`: 라벨 구조 확인 후 PPE worker에 연결
- 목요일 현장 보호구 데이터는 **착용/미착용/일반안경/보안경/마스크** 조건을 다양하게 모아 학습/검증으로 분리합니다.
